<a href="https://colab.research.google.com/github/Glaze0/Assignment/blob/main/Python_TypeHints_%26_Data_Validation_with_Pydantic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Problem Statement:** Build a validator that rejects bad payment JSON at the door, with a clear error message saying exactly what was wrong.

In [ ]:
def charge(amount: int) -> str:
    return f"Charging ₹{amount}"

print(charge("-500 rupees"))

Charging ₹-500 rupees


In [ ]:
# {"order_id": "ord_101", "amount": 2500, "items" : [i1, i2, i3]}

# {"order_id": "ord_102", "amount": -500, "items" : [i1, i2, i3]}

In [ ]:
def charge(amount):
    return f"Charging ₹{amount}"

print(charge("-500 rupees"))

Charging ₹-500 rupees


In [ ]:
def charge(amount: int) -> str:
    return f"Charging ₹{amount}"

charge(20)

'Charging ₹20'

In [ ]:
charge("-alpha")

'Charging ₹-alpha'

In [ ]:
merchant: str = "Chai Point"
amount: int = 250
is_open: bool = True

print(merchant)
print(amount)
print(is_open)

Chai Point
250
True


In [ ]:
def charge(amount: int, currency: str = "INR"):
    return f"Charging {currency} {amount}"

In [ ]:
charge(50, "$")

'Charging $ 50'

In [ ]:
charge("tweenty")

'Charging INR tweenty'

In [ ]:
def get_order_ids(merchant: str)-> list[str]:
    return ["ord_101", "ord_102"]

In [ ]:
get_order_ids("chai point")

['ord_101', 'ord_102']

In [ ]:
prices: dict[str, int] = {"chai" : 30, "samosa" : 50}

In [ ]:
print(prices)

{'chai': 30, 'samosa': 50}


In [ ]:
from typing import Optional

def find_coupon(code: str)-> Optional[str]: # str or None
    return None

In [ ]:
# Q- What two kinds of values does Optional[int] allow?

## Pydantic

In [ ]:
from pydantic import BaseModel

In [ ]:
class Payment(BaseModel):
    order_id: str
    amount: int
    currency: str = "INR"

In [ ]:
p1 = Payment(order_id="ord101", amount = 2500)
print(p1)
print(p1.order_id, p1.currency)

order_id='ord101' amount=2500 currency='INR'
ord101 INR


In [ ]:
p2 = Payment(order_id="ord102", amount = -50)
print(p2)

order_id='ord102' amount=-50 currency='INR'


In [ ]:
p3 = Payment(order_id="ord102", amount = "200")
print(p3)

order_id='ord102' amount=200 currency='INR'


In [ ]:
type(p3.amount)

int

In [ ]:
int("200")

200

In [ ]:
p4 = Payment(order_id="ord102", amount = "fifty")
print(p4)

ValidationError: 1 validation error for Payment
amount
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='fifty', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

In [ ]:
int("5,600")

ValueError: invalid literal for int() with base 10: '5,600'

In [ ]:
int("5600")

5600

In [ ]:
from pydantic import BaseModel, Field

class Payment(BaseModel):
    order_id: str = Field(min_length=3)
    amount: float = Field(gt=0, le=500000)
    email: str = Field(pattern=r".+@.+" ) # looking for a valid email.


In [ ]:
# "mohit@paypal.com"
# "mohit.gmail"

In [ ]:
from pydantic import ValidationError

try:
    p = Payment(order_id="o1", amount = -30, email="hello.gmail.com")
    print(p)
except ValidationError as e:
    print(e)

print("\n\n100 other lines of code...")

3 validation errors for Payment
order_id
  String should have at least 3 characters [type=string_too_short, input_value='o1', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short
amount
  Input should be greater than 0 [type=greater_than, input_value=-30, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than
email
  String should match pattern '.+@.+' [type=string_pattern_mismatch, input_value='hello.gmail.com', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_pattern_mismatch


100 other lines of code...


In [ ]:
try:
    p = Payment(order_id="o1", amount = -30, email="hello.gmail.com")
    print(p)
except ValidationError as e:
    for err in e.errors():
        print(err['loc'] ,"|", err['msg'])

('order_id',) | String should have at least 3 characters
('amount',) | Input should be greater than 0
('email',) | String should match pattern '.+@.+'


In [ ]:
try:
    p = Payment(order_id="ord_12", amount=300, email="hello@gmail.com")
    print(p)
except ValidationError as e:
    print(e)

order_id='ord_12' amount=300 email='hello@gmail.com'


In [ ]:
class Customer(BaseModel):
    name: str
    email: str = Field(pattern=r".+@.+" ) # looking for a valid email.

class Payment(BaseModel):
    order_id: str
    amount: float = Field(gt=0)
    customer: Customer

In [ ]:
raj = Customer(name="raj", email="raj@gmail.com")
raj

Customer(name='raj', email='raj@gmail.com')

In [ ]:
type(raj)

__main__.Customer

In [ ]:
pay = Payment(order_id ="o111", amount=200, customer=raj)
pay

Payment(order_id='o111', amount=200.0, customer=Customer(name='raj', email='raj@gmail.com'))

In [ ]:
print(pay)

order_id='o111' amount=200.0 customer=Customer(name='raj', email='raj@gmail.com')


In [ ]:
pay.customer.email

'raj@gmail.com'

In [ ]:
pp = {
    'order_id' : "o212",
    'amount' : 50,
    'customer' : {'name' : "vikas", 'email': "vikas@gmail.com"}
}
pp

{'order_id': 'o212',
 'amount': 50,
 'customer': {'name': 'vikas', 'email': 'vikas@gmail.com'}}

In [ ]:
Payment.model_validate(pp)

Payment(order_id='o212', amount=50.0, customer=Customer(name='vikas', email='vikas@gmail.com'))

In [ ]:
import json
request_data = json.dumps(pp)
request_data

'{"order_id": "o212", "amount": 50, "customer": {"name": "vikas", "email": "vikas@gmail.com"}}'

In [ ]:
pay = Payment.model_validate_json(request_data)
pay

Payment(order_id='o212', amount=50.0, customer=Customer(name='vikas', email='vikas@gmail.com'))

In [ ]:
pay.amount

50.0

In [ ]:
pay

Payment(order_id='o212', amount=50.0, customer=Customer(name='vikas', email='vikas@gmail.com'))

In [ ]:
pay.model_dump_json()

'{"order_id":"o212","amount":50.0,"customer":{"name":"vikas","email":"vikas@gmail.com"}}'

In [ ]:
# Q - True or False: model_dump_json() returns a dictionary.

In [ ]:
a = "mohit"
type(a)

str

In [ ]:
b = "mohit123"
type(b)

str

## Mini Project

In [ ]:
import json
from pydantic import BaseModel, Field, ValidationError

In [ ]:
class Customer(BaseModel):
    name: str = Field(min_length=2)
    email: str = Field(pattern=r".+@.+" ) # looking for a valid email.

class Payment(BaseModel):
    order_id: str =  Field(min_length=2)
    amount: float = Field(gt=0)
    currency: str = "INR"
    customer: Customer

In [ ]:
incoming = [
    '{"order_id": "ord_101", "amount": 2500, "customer": {"name": "Priya", "email": "priya@shop.com"}}',
    '{"order_id": "ord_102", "amount": "4999", "customer": {"name": "Ravi", "email": "ravi@shop.com"}}',
    '{"order_id": "ord_103", "amount": -500, "customer": {"name": "Hacker", "email": "bad-email"}}',
    '{"order_id": "ord_104", "customer": {"name": "Meera", "email": "meera@shop.com"}}',
    '{"order_id": "ord_104", "amount": 45322, "customer": {"name": "Harsh", "email": "harsh@shop.com"}}',
    '{"order_id": "ord_106", "amount": 19065, "customer": {"name": "Amit", "email": "amit@shop.com"}}',
]

In [ ]:
accepted = []
rejected = []

for raw in incoming:
    try:
        pay = Payment.model_validate_json(raw)
        # print(pay)
        accepted.append(pay)
    except ValidationError as e:
        for err in e.errors():
            print(err['loc'] ,"|", err['msg'])

        rejected.append(pay)

('amount',) | Input should be greater than 0
('customer', 'email') | String should match pattern '.+@.+'
('amount',) | Field required


In [ ]:
accepted

[Payment(order_id='ord_101', amount=2500.0, current='INR', customer=Customer(name='Priya', email='priya@shop.com')),
 Payment(order_id='ord_102', amount=4999.0, current='INR', customer=Customer(name='Ravi', email='ravi@shop.com')),
 Payment(order_id='ord_104', amount=45322.0, current='INR', customer=Customer(name='Harsh', email='harsh@shop.com')),
 Payment(order_id='ord_106', amount=19065.0, current='INR', customer=Customer(name='Amit', email='amit@shop.com'))]

In [ ]:
print(f"Accepted: {len(accepted)}    Rejected: {len(rejected)}")


Accepted: 4    Rejected: 2


In [ ]:
print("\n--- Accepted ---")
for p in accepted:
    print(f"{p.order_id}: ₹{p.amount} from {p.customer.name}")



--- Accepted ---
ord_101: ₹2500.0 from Priya
ord_102: ₹4999.0 from Ravi
ord_104: ₹45322.0 from Harsh
ord_106: ₹19065.0 from Amit


In [ ]:
total = 0
for p in accepted:
    total += p.amount

print(f"\nSettlement total: ₹{total}")


Settlement total: ₹71886.0


##Build a Swiggy Order Validator:

- MenuItem model: name (min 2 chars), price (greater than 0), quantity (1 to 20)

- Order model: order_id, customer_name, items: list[MenuItem], and coupon: Optional[str] = None

- Validate a list of 4 raw JSON orders where at least two are broken in different ways

- Print accepted orders with their totals, print rejected ones with field-level reasons, and save the accepted orders using model_dump_json()